In [ ]:
%pip install anthropic python-dotenv

In [2]:
# load env
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# create client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"
 

In [ ]:
# make request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role" : "user",
            "content" : "What is Quantum Computing?"
        }
    ]
)

In [ ]:
message.content[0].text

In [ ]:
# claude do not have previous conversation memory
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role" : "user",
            "content" : "Write another sentence?"
        }
    ]
)

In [ ]:
message.content[0].text

In [ ]:
# Helper to store the previous conversation
def add_user_message(messages, text):
    user_messages = {"role" : "user", "content" : text}
    messages.append(user_messages)

def add_assistant_message(messages, text):
    assistant_messages = {"role" : "assistant", "content" : text}
    messages.append(assistant_messages)

def chat(messages, system=None, temperature=1.0, stop_squences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_squences
    }

    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# test temperature
messages = []

# Add intial user message
add_user_message(messages, "generate a one sentence movie idea?")
messages

answer = chat(messages, temperature=0.0)
answer

In [ ]:
# List messages
messages = []

# Add intial user message
add_user_message(messages, "What is Quantum Computing?")
messages

answer = chat(messages)
answer

# Take the answer and add it to the conversation
add_assistant_message(messages, answer)
messages

# Add user follow-up message
add_user_message(messages, "Write another sentence?")

final_answer = chat(messages)
final_answer

In [ ]:
# Initial message
messages = []

# while true loop to run the chatbot forever
while True:
    user_input = input("User:")
    print("User input: ", user_input)

    add_user_message(messages, user_input)
    answer = chat(messages)
    print("Claude: ", answer)

    add_assistant_message(messages, answer)
    answer = chat(messages)
    print("Claude: ", answer)

In [ ]:
# system prompt
messages = []

system = """
you are a patient math tutor.
Do not directly answer the question. Instead, ask the student questions to guide them to the answer."""

add_user_message(messages, "How do I solver 5x + 3 = 8 for x?")
answer = chat(messages,system=system)

In [ ]:
# streaming response
messages = []

add_user_message(messages, "generate a one sentence movie idea?")
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

In [ ]:
# streaming response best approach
messages = []

add_user_message(messages, "generate a one sentence movie idea?")
with client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
) as stream:
    for text in stream.text_stream:
        print(text, end="")

In [ ]:
# streaming response final answer
messages = []

add_user_message(messages, "generate a one sentence movie idea?")
with client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

    stream.get_final_message()

In [ ]:
# stop claude to send header and footer
messages = []
add_user_message(messages, "generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequence=["```"])
text

In [ ]:
import json
json.loads(text.strip())